# Pivot and Graph Functions [initial draft complete]

By Kenneth Burchfiel

Released under the MIT License

This notebook will demonstrate how to use functions to simplify the Plotly graphing process. These functions will also prove useful within the online visualizations section of Python for Nonprofits.

As shown in the 'graphing' notebook within this section, pivot tables play a crucial role in generating Plotly charts. They allow you to turn a raw dataset into a table of aggregate values that can easily be rendered as a bar graph, line graph, or alternative plot type. Thanks to the powerful pandas library, these tables are easy to produce within Python.

However, creating separate pivot tables for each graph still requires a decent amount of work. In addition, there are times when coding these pivot tables would be impractical. Suppose, for instance, that you're putting together an online dashboard that visualizes mean survey result scores in bar chart form. You would like to allow users to compare these scores by up to 6 potential variables: Starting Year, Season, Gender, Graduation Year, College, and Level. (In other words, they should be able to view these results by Season and College; College, Level, and Season; all 6 variables; and even no variables at all--in which case all results will be grouped into a single bar.)

If you wanted to create a pivot table for each of these comparison options, you'd end up having to construct *at least* 64 pivot tables. (There are 6 comparison options, each of which can either be present or absent--resulting in a total of 2^6 = 64 tables.) However, when you take the color variable option into account, you might need to produce up to 256 tables. In addition, if we consider x variable order to matter, the number of tables that must be created **will easily exceed 1,000**.*

In order to avoid creating hundreds or even thousands of pivot tables for this online dashboard, we'll need to create **an 'autopivot' function that, given a list of comparison variables, a y variable, and a color variable, returns a pivot table, x and y variables, and a color variable.** We will create that very function within this notebook. 

Although this pivot table function will play a pivotal (pun intended) role in our final graph, we'll also need to define **an 'autobar' function that converts the output of the autopivot function into a graph.** This notebook will also show how to produce that function. (We could add this code into our main pivot table function, but separating the two allows other graph types--such as line charts--to be created rather than bar charts, thus making the code more versatile.)

\*The spreadsheet I used for these calculations is present within color_combination_possibilities.xlsx. These can be analyzed as combination and permutation problems, although the presence of a color variable adds a bit more complexity. To be honest, I'm not certain about these calculations, but it's fair to say that the number of possible pivot tables (and charts) that could be created with up to 6 comparison options is . . . *a lot.*



In [ ]:
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine
e = create_engine('sqlite:///../Appendix/nvcu_db.db')

# Creating a detailed survey results dataset:

This dataset will include data from both our survey results *and* enrollment tables, thus allowing for additional comparison options.

### Importing each dataset:

In [ ]:
df_enrollment = pd.read_sql(
    'select * from curr_enrollment', con = e)
df_enrollment

In [ ]:
df_survey_results = pd.read_sql(
    'select * from survey_results', con = e)
df_survey_results

### Merging these datasets together:

In [ ]:
df_se = df_survey_results.merge(
    df_enrollment, on = 'student_id', how = 'left')
# The 'se' in df_se stands for 'survey and enrollment.'

# Removing columns that we're not interested in using for our 
# comparisons:
df_se.drop(
    ['student_id', 'first_name', 'last_name',
     'date_of_birth', 'matriculation_number'], 
    axis = 1, inplace = True)

# Reformatting our column names so that they'll show up better
# within graphs):

df_se.columns = [
    column.replace('_', ' ').title() 
    for column in df_se.columns]

df_se['Season For Sorting'] = (
    df_se[
'Season'].map({'Fall':0,'Winter':1,'Spring':2}))
df_se.sort_values(
    ['Season For Sorting', 'Level For Sorting'],
    inplace = True)

df_se.head()


df_se is an excellent candidate for creating a diverse set of pivot tables and charts because it represents [microdata](https://en.wikipedia.org/wiki/Microdata_(statistics)): each respondent has his or her own row. This allows you to calculate mean statistics for an arbitrary set of comparison variables without running into the 'average of averages' problem.* However, you may not always have access to this type of data. Instead, you might receive an *aggregated* dataset in which averages by different groups are pre-calculated. df_se_pivot, defined below, shows one example.

However, if this table also shows the number of samples in each row (as the 'Responses' column in df_se_pivot does), you can still use this dataset to calculate accurate mean values. The autopivot function will demonstrate how.

### A primer on the 'average of averages' problem

\*This problem arises when differences in group sizes cause skew a calculated average that's based on averages for those groups. You can avoid this issue by adjusting your calculations for group sizes.

Here's an example: Suppose you're asked to calculate the average spring survey result for students across two colleges. One college has an average result of 70, and the other has an average result of of 98.*

A naive approach would be to calculate the average as (98 + 70 / 2) = 84. This 'average of averages' would indeed be accurate if both colleges had the same number of students. However, suppose that the college with a result of 70 had 6 times as many students as the other! In this case, the correct average--weighted at the student level, not the college level--would be ((70 * 6) + (98 * 1))/7, or 74.*

When analyzing aggregated data, you can avoid the 'averages of averages' problem by making sure to weight your calculations by a group size metric (response counts, test counts, etc). If such data aren't available, your best option might be to report an 'average of averages' while also explaining why this average may be inaccurate.

In [ ]:
df_se['Responses'] = 1
df_se_pivot = df_se.pivot_table(
    index = ['Starting Year', 'Season', 'Gender', 
             'Matriculation Year', 'College', 'Class Of',
             'Level', 'Level For Sorting', 'Season For Sorting'],
    values = ['Score', 'Responses'],
    aggfunc = {'Score':'mean', 'Responses':'sum'}).reset_index()
df_se_pivot

## A basic autopivot function

The following code shows a simple version of an 'autopivot' function that can automatically create both pivot tables to be graphed and corresponding graphing variables. One key variable generated by this function is a field that groups together all of the individual comparison values passed to x_vars[] and can thus serve as the 'x' argument within a Plotly graph call.

This function has a number of limitations: for instance, it doesn't take color options into account, and it also can't accommodate variables that should be included in the *pivot table* function (e.g. for sorting purposes) but excluded from the *x variable column*. However, I chose to include it here in order to highlight the core concepts of this 'autopivot' function clearer. (The final autopivot function, shown later in this notebook, gets a bit more complicated.)



In [ ]:
def autopivot_simple(df, y, aggfunc, x_vars = [],
                    overall_data_name = 'All Data'):
    '''This is a simple version of an 'autopivot' chart. Given a 
    DataFrame, a y variable, an aggregate function, and an arbitrary list 
    of variables, it will return a pivot table, an x variable, 
    and a y variable that can then get fed into a graphing function.

    For more documentation on these steps, 
    see the full autopivot function below. (I excluded that documentation
    from this code in order to prevent redundancy.)
    '''

    df_for_pivot = df.copy()
    df_for_pivot

    # Handling a situation in which no comparion variables were provided:
    if len(x_vars) == 0: 
        df_for_pivot[overall_data_name] = overall_data_name
        df_pivot = df_for_pivot.pivot_table(
            index = overall_data_name, values = y, 
            aggfunc = aggfunc).reset_index()
        x_val_name = overall_data_name
        color = overall_data_name
        barmode = 'relative'
        index = [overall_data_name]         
       
    else:
        print("x_vars:",x_vars)    

        # Creating a pivot table using the arguments provided:
        df_pivot = df_for_pivot.pivot_table(
            index = x_vars, values = y, aggfunc = aggfunc).reset_index()
        
        # Creating the x variable's column name:
        # (This name can then get fed into a graphing function.)
        x_val_name = ('/').join(x_vars)

        # Creating the values for this column:
        # (The function does so by combining each row's values for 
        # the fields represented in x_vars into a single string.)
        df_pivot[x_val_name] = df_pivot[x_vars[0]].astype('str')
        for i in range(1, len(x_vars)):
            df_pivot[x_val_name] = (
                df_pivot[x_val_name] 
                + '/' + df_pivot[x_vars[i]].astype('str'))
            
    return df_pivot, x_val_name, y, aggfunc

## A basic autobar function

The following function uses the output of autopivot_simple as inputs for a bar chart. It's quite limited relative to the final autobar function that will be shown later in this code, but it still demonstrates the utility of the autobar_simple function.

In [ ]:
def autobar_simple(df_pivot, x_val_name, y, aggfunc):
    '''This function creates a bar graph of a pivot table (such as one
    created by autopivot_simple.
    For more documentation on this function, view the full autobar()
    function, which will be defined below.
    '''
    
    fig = px.bar(df_pivot, x = x_val_name, y = y, text_auto = '.0f',
                title = f"{aggfunc.title()} {y} by {x_val_name}") 
    return fig

## Testing out autopivot_simple and autobar_simple:

We'll first use these functions to graph average survey results by level and season:

In [ ]:
df_pivot, x_val_name, y, aggfunc = autopivot_simple(
    df_se, y = 'Score', aggfunc = 'mean', x_vars = ['Level', 'Season'])

print(x_val_name, y, df_pivot.head())
fig = autobar_simple(
    df_pivot = df_pivot, x_val_name = x_val_name, y = y,
aggfunc = aggfunc)                      

fig

Next, we'll display these averages by season, college, and level:

In [ ]:
df_pivot, x_val_name, y, aggfunc = autopivot_simple(
    df_se, y = 'Score', aggfunc = 'mean', x_vars = [
        'Season', 'College', 'Level'])

fig = autobar_simple(
    df_pivot = df_pivot, x_val_name = x_val_name, y = y,
    aggfunc = aggfunc
    )                      

fig

Finally, we'll create a chart that groups all data together:

In [ ]:
df_pivot, x_val_name, y, aggfunc = autopivot_simple(
    df_se, y = 'Score', aggfunc = 'mean', x_vars = [])

fig = autobar_simple(
    df_pivot = df_pivot, x_val_name = x_val_name, y = y, aggfunc = aggfunc
    )                      

fig

The above examples demonstrate how the autopivot_simple and autobar_simple functions can work together to generate all sorts of different table and chart combinations, thus eliminating the need to write pivot table and bar chart code for each set of comparison variables a user might want to analyze. 

However, the charts they work together to produce leave much to be desired. There are no colors: the titles could use some work; and levels are sorted alphabetically rather than chronologically. Therefore, we'll now create more robust versions of these functions.

### Defining an improved autopivot() function:

In [ ]:
def autopivot(df, y, aggfunc, x_vars = [], 
                     color = None, x_vars_to_exclude = [],
                    overall_data_name = 'All Data',
             weight_col = None):
    '''This function will create a pivot table of df that can be used within
    a Plotly graph. It will also return x, y, color, and barmode variables
    that can get incorporated within Plotly charts. (Storing the charting
    code within a separate function makes autopivot more versatile,
    as its output can then be used as the basis for multiple chart types.)

    This function has been designed to work with varying lengths of x_vars
    (i.e. multiple numbers of x variables), including a length of 0. 
    It should therefore prove useful for applications, such as interactive 
    dashboards, that allow the user
    to choose an arbitrary number of comparison variables for a graph.

    df: the DataFrame to use as the basis for the pivot table.
    
    y: the y variable to use within the Plotly graph.
    
    aggfunc: the aggregate function to use within the pivot_table() call
    (e.g. 'mean', 'count', etc.).
    
    x_vars: a list of comparison variables to be converted into 
    an x variable. Pass an empty list in order to group all y values together.
    
    color: the variable that should serve as the color argument for the Plotly
    graph. 

    x_vars_to_exclude: variables that will be incorporated into the pivot table
    (e.g. to ensure it's sorted correctly) but should not be present in the 
    final chart. 
    (For instance, if you've added 'Season' as an x_vars entry or
    color variable, and its values are 'Fall', 'Winter', and 'Spring,' 
    you may also want to add a 'Season For Sorting' column with values of 0, 1, 
    and 2 for Fall, Winter, and Spring, respectively so that they'll 
    appear in chronological order within your chart. These variables should 
    also be placed in x_vars_to_exclude so that they won't make your 
    final graph more complicated and/or cluttered than it needs to be.

    If a variable within x_vars_to_exclude isn't present in your x_vars list
    or your color argument, it will get ignored by the function and thus
    shouldn't cause any issues.
    
    overall_data_name: When x_vars is empty, autopivot will group all
    data into a single row. overall_data_name specifies the name that you 
    would like to give to this data point.    

    weight_col: A column containing group sizes that, if included, will
    be used to calculate weighted averages. (If weight_col is None,
    weighted averages will *not* be calculated.)
    '''

    # Creating a copy of the initial dataset in order to ensure that the 
    # following code does not modify it:
    df_for_pivot = df.copy()
    df_for_pivot

    # Determining which x variables will appear in the final chart:
    x_vars_for_chart = list(set(x_vars) - set(x_vars_to_exclude))
    
    x_var_count = len(x_vars_for_chart)


    if weight_col is not None: # In this case, weighted averages
        # will be calculated.
        # Multiplying each y value by its corresponding weight in order
        # to allow weighted averages to be calculated:
        df_for_pivot[f'{y}_*_{weight_col}'] = (
            df_for_pivot[y] * df_for_pivot[weight_col])
        # These 'y_*_weight' values will get added together during the 
        # pivot table call, as will the corresponding weight column values.
    
    if x_var_count == 0: # Because no comparison variables will appear in
        # the final chart, the function will instead group all data together. 
        # It will do so
        # by creating a new column that has the same value for each row.
        # This column can then serve as the index for the pivot function as well
        # as the color value.
        df_for_pivot[overall_data_name] = overall_data_name

        if weight_col is not None: # In this case, a weighted average
            # of all data within the table will be created.
            df_pivot = df_for_pivot.pivot_table(
            index = overall_data_name, 
                values = [f'{y}_*_{weight_col}', weight_col], 
            aggfunc = 'sum').reset_index()
            # Calculating the weighted average of y by dividing
            # the y_*_weight_col field by its corresponding group size:
            df_pivot[y] = df_pivot[f'{y}_*_{weight_col}'] / df_pivot[weight_col]
        
        else:
            df_pivot = df_for_pivot.pivot_table(
                index = overall_data_name, values = y, 
                aggfunc = aggfunc).reset_index()
        x_val_name = overall_data_name
        color = overall_data_name
        barmode = 'relative'
        index = [overall_data_name] # This value won't be used
        # within autopivot() (defined below), but it will still
        # get created here in order to prevent scripts that expect it to be
        # returned from crashing.
        
       
    else:    
        # If the color variable is also present in x_vars and more than one variable
        # is present within the set of x vars to be charted*, 
        # the graph will display redundant data. Therefore, 
        # the function will remove this variable from x_vars below.
        # (If the only x variable to be charted is also the color variable,
        # we won't want to remove it from x_vars; otherwise, we'd end up with an 
        # empty list of x variables.)
        # *This set (defined above as x_vars_for_chart) excludes any variables
        # also present in x_vars_to_exclude. This will prevent the function
        # from removing a color variable from x_vars that would have been the
        # only variable left in x_vars following the removal of 
        # the variable to exclude (which would cause the function to crash).
        
        # (For example: suppose our color variable were 'Season'; 
        # our x_vars contents were ['Season', 'Season_for_Sorting'];
        # and our x_vars_to_exclude list were ['Season_for_Sorting']. If
        # we chose to remove the color variable from x_vars contents
        # because it contained more than one variable, we'd end up with
        # an empty x_vars list once 'Season_for_Sorting' got removed. Removing
        # this variable to exclude from our list of x vars to pass to len() below
        # will prevent this error.
    
        
        if (color in x_vars) and (len(x_vars_for_chart) > 1):
            x_vars.remove(color)
    
        print("x_vars:",x_vars)
        
        # Initializing a list of variables to be passed to the 'index' argument
        # within the pivot_table call:
        index = x_vars.copy()
        
        # We'll want to make sure to include the color variable in the pivot index
        # as well so that it can be accessed by the charting code.
        if color is not None and color not in index:
            index.append(color)
    
        print("index prior to pivot_table() call:",index)
        

        if weight_col is not None: # In this case, weighted averages
            # will be calculated.
            df_pivot = df_for_pivot.pivot_table(
                index = index, values = [f'{y}_*_{weight_col}', weight_col], 
                aggfunc = 'sum').reset_index()           
            df_pivot[y] = df_pivot[f'{y}_*_{weight_col}'] / df_pivot[weight_col]
            
        else:
            df_pivot = df_for_pivot.pivot_table(
                index = index, values = y, aggfunc = aggfunc).reset_index()
    
        # Now that the pivot table has been created, the variables
        # in x_vars_to_exclude can be removed from our x_vars list, our 
        # index, and our DataFrame, thus preventing our final 
        # graphs from being too cluttered.
        for var_to_exclude in x_vars_to_exclude:
           if var_to_exclude in x_vars:
               x_vars.remove(var_to_exclude)
               index.remove(var_to_exclude)
            # These variables could get removed from df_pivot also,
            # but the user might find them helpful for future sorting needs--
            # so they'll be retained for now.
            # df_pivot.drop(var_to_exclude, axis = 1, inplace = True)

        # Determining the name of the x value column by
        # joining together all of the values in x_vars that will get
        # incorporated into its own values:
        x_val_name = ('/').join(x_vars)

        # Initializing the x_val_name column values (which will serve
        # as the x axis entries within Plotly charts) by 
        # converting the first item within x_vars to a string,
        # then adding other string-formatted values to it:
        # (Slashes will separate these various values.)
        df_pivot[x_val_name] = df_pivot[x_vars[0]].astype('str')
        for i in range(1, len(x_vars)):
            df_pivot[x_val_name] = (
                df_pivot[x_val_name] 
                + '/' + df_pivot[x_vars[i]].astype('str'))
    
        
        # If there are fewer than two unique variables to be graphed, we'll want
        # to set the barmode argument to 'relative' so that, in the event we create
        # a bar chart, the bars won't be far apart from one another. If there are
        # two or more unique variables, we can use 'group' instead.
        # (If there's a single variable within x_vars that matches the 'color' 
        # variable, we will also want to set barmode to 'relative,' since only
        # one variable will actually be displayed on the graph. Therefore, 
        # I added in a set() call in the following code; set() keeps only unique
        # instances of a list and will thus prevent the same x_vars variable
        # and color variable from being counted as two variables within
        # this if/else statement.
        
        if color is not None:
            unique_graphed_vars = set(x_vars + [color])
        else:
            unique_graphed_vars = set(x_vars)
        
        if len(unique_graphed_vars) < 2:
            barmode = 'relative' 
        else:
            barmode = 'group'
    
    return df_pivot, x_val_name, y, color, barmode, x_var_count, index, aggfunc

### Defining an improved autobar() function:

In [ ]:
def autobar(df_pivot, x_val_name, y, 
                             color, barmode, x_var_count, index, aggfunc):
    '''This function creates a bar graph of a pivot table (such as one
    created within autopivot(). 
    Note that the arguments for this function
    correspond to the values returned by autopivot(); more information on
    each can be found within that function.
    '''
    
    # Creating a title for the chart:
    # Suppose our y value is 'Score' and our aggregate function is 'mean.' 
    # If our original x variable count was 0, our title can simply be 
    # 'Overall Mean Score.' If we had just one graph variable ('College') 
    # and no color variable, our title could be 'Mean Score by College.'
    # If we also had a 'Level' color variable, our title
    # could be 'Mean Score by College and Level'. Finally, if we added
    # another x variable ('Season' to our list, our mean title could be
    # 'Mean Score by College, Season, and Level.'
    # The following code includes four title definitions to cover
    # these four scenarios. (Note that 'index' is used rather than 'x_vars'
    # because the former variable includes both our x variables and (if present)
    # our color variable, and both of these should be incorporated into the title.
    
    if x_var_count == 0: 
        plot_title = f"Overall {aggfunc.title()} {y}"
    elif len(index) == 1:
        plot_title = f"{aggfunc.title()} {y} by {index[0]}"
    elif len(index) == 2:
        plot_title = f"{aggfunc.title()} {y} by {index[0]} and {index[1]}" 
    else:
        plot_title = f"{aggfunc.title()} {y} by {(', ').join(
            index[0:-1])}, and {index[-1]}" 
        
    # The following code will still work if color is set to None.
    fig = px.bar(df_pivot, x = x_val_name, y = y, 
           color = color, barmode = barmode,
           text_auto = '.1f', title = plot_title)
    if x_var_count == 0: # In this case, the x axis tick, x axis title, and
        # legend entry will all be the same, so we can hide two of those elements.
        fig.update_layout(showlegend = False,
                         xaxis_title = None)
    return fig

In [ ]:
def autopivot_plus_bar(
    df, y, aggfunc, x_vars = [], color = None, 
    x_vars_to_exclude = [], overall_data_name = 'All Data',
    weight_col = None):
    '''This function calls both autopivot() and autobar(), thus simplifying
    the process of using both functions within a script. 
    See autopivot() and autobar()'s individual function definitions for more
    documentation on each.'''
        
    df_pivot, x_val_name, y, color, barmode, x_var_count, \
    index, aggfunc = autopivot(
        df = df, y = y, aggfunc = aggfunc, 
        x_vars = x_vars, color = color, 
        x_vars_to_exclude = x_vars_to_exclude,
        overall_data_name = overall_data_name, weight_col = weight_col)
    
    fig_bar = autobar(
        df_pivot = df_pivot, x_val_name = x_val_name, y = y, 
        color = color, barmode = barmode, x_var_count = x_var_count, 
        index = index, aggfunc = aggfunc)

    return fig_bar



## Demonstrating these functions

First, here's a look at the output of autopivot() on its own:

In [ ]:
(df_pivot, x_val_name, y, color, barmode, 
 x_var_count, index, aggfunc) = autopivot(
    df = df_se, y = 'Score', 
    x_vars = ['College', 'Level For Sorting', 'Level'], 
    color = 'Season', 
    x_vars_to_exclude = ['Level For Sorting', 'Season For Sorting'],
    aggfunc = 'mean')
print(x_val_name, y, color, barmode)
df_pivot.head(5)

And here's what autobar() produces when the above autopivot() output serves as its arguments:

In [ ]:
fig = autobar(
    df_pivot = df_pivot,
    x_val_name = x_val_name, y = y,
    color = color, barmode = barmode, 
    x_var_count = x_var_count, index = index,
    aggfunc = aggfunc)
fig
                               

We can simplify this code even further by calling autopivot_plus_bar():

In [ ]:
autopivot_plus_bar(
    df = df_se, y = 'Score', 
    x_vars = ['College', 'Level For Sorting', 'Level'], 
    color = 'Season', 
    x_vars_to_exclude = ['Level For Sorting', 'Season For Sorting'],
    aggfunc = 'mean')

Here's what the output looks like with 3 comparison variables:

In [ ]:
(df_pivot, x_val_name, y, color, barmode, 
 x_var_count, index, aggfunc) = autopivot(
    df = df_se_pivot, y = 'Score', 
    x_vars = ['College', 'Level For Sorting', 'Level'], 
    color = 'Season', 
    x_vars_to_exclude = ['Level For Sorting', 'Season For Sorting'],
    aggfunc = 'mean', weight_col = 'Responses')
print(x_val_name, y, color, barmode)
print(df_pivot.head(5))

fig = autobar(
    df_pivot = df_pivot,
    x_val_name = x_val_name, y = y,
    color = color, barmode = barmode, 
    x_var_count = x_var_count, index = index,
    aggfunc = aggfunc)
fig
                               

Here's a chart with just 2 comparison variables:

In [ ]:
autopivot_plus_bar(
    df = df_se, y = 'Score', 
    x_vars = ['Level', 'Season For Sorting'], 
    color = 'Season', 
    x_vars_to_exclude = ['Season For Sorting'],
    aggfunc = 'mean')

Note that the above output is identical to the output of the following cell, which uses an aggregated dataframe (df_se_pivot) as its data source. This identical output was achieved by adding in a weight column ('Respones').

In [ ]:
autopivot_plus_bar(
    df = df_se_pivot, y = 'Score', 
    x_vars = ['Level'], 
    color = 'Season', 
    x_vars_to_exclude = ['Season For Sorting'],
    aggfunc = 'mean', weight_col = 'Responses')                              

Note that the following output, which uses df_se_pivot as well but does *not* reference a weight column, differs slightly from the above two graphs. This is because it is falling into the 'average of averages' error noted earlier.

In [ ]:
autopivot_plus_bar(
    df = df_se_pivot, y = 'Score', 
    x_vars = ['Level'], 
    color = 'Season', 
    x_vars_to_exclude = ['Season For Sorting'],
    aggfunc = 'mean')                              

These functions also allow the color argument to be skipped, though the result is rather plain:

In [ ]:
autopivot_plus_bar(
    df = df_se, y = 'Score', 
    x_vars = ['College', 'Level'],
    aggfunc = 'mean')

They also allow an empty list of comparison values to get passed to x_vars, thus producing a single bar that represents the average of all values in the DataFrame.

In [ ]:
autopivot_plus_bar(
    df = df_se, y = 'Score', 
    x_vars = [], 
    color = None, 
    x_vars_to_exclude = [],
    aggfunc = 'mean')                               

At the opposite extreme, we can graph 6 comparison variables within the same chart (though the result is not the easiest to read):

In [ ]:
autopivot_plus_bar(
    df = df_se, y = 'Score', 
    x_vars = [
        'Starting Year', 'Season', 'Gender', 'College', 'Class Of', 
        'Level For Sorting', 'Level'], 
    color = 'Season', 
    x_vars_to_exclude = ['Level For Sorting'],
    aggfunc = 'mean')                               

The autopivot() and autobar() functions defined here can help speed up the process of creating standalone Plotly charts. However, these functions will prove even more useful within the Online Visualizations section of Python for Nonprofits--as they'll allow us to turn a small set of user inputs into a wide variety of charts and tables.